# Sarvam-30B → GGUF Conversion Attempt

**One-click attempt** to convert `sarvamai/sarvam-30b` to GGUF for Ollama/llama.cpp/LM Studio.

Sarvam uses **sigmoid routing** (not softmax) in its MoE. This may not be supported yet.

**Instructions:** Runtime → Change runtime type → GPU (T4 is fine) → **Run All**

Read: [Sarvam. Open is not sovereign](https://mtrajan.substack.com/p/sarvam-open-is-not-sovereign)

## Step 1: Setup + Disk Strategy

In [ ]:
!pip install -q huggingface_hub hf_transfer sentencepiece protobuf transformers numpy gguf

# === Always mount Google Drive ===
# Sarvam-30B is ~60GB in safetensors. With HF cache overhead,
# Colab's local disk (~100GB on T4, ~200GB on A100) is too tight.
# Google Drive gives us reliable overflow storage.

from google.colab import drive
drive.mount('/content/drive')

import os, shutil

MODEL_DIR = "/content/drive/MyDrive/sarvam-gguf/sarvam-30b"
OUTPUT_DIR = "/content/drive/MyDrive/sarvam-gguf/output"
LLAMA_CPP_DIR = "/content/llama.cpp"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# HF cache on local disk (fast SSD, expendable)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

total, used, free = shutil.disk_usage("/content")
print(f"Local disk: {free/(1024**3):.1f} GB free")
total_d, used_d, free_d = shutil.disk_usage("/content/drive/MyDrive")
print(f"Google Drive: {free_d/(1024**3):.1f} GB free")
print(f"\nModel dir:  {MODEL_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'No GPU'

## Step 2: Build llama.cpp

In [ ]:
import os
LLAMA_CPP_DIR = "/content/llama.cpp"

if not os.path.exists(LLAMA_CPP_DIR):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_CPP_DIR}
else:
    !cd {LLAMA_CPP_DIR} && git pull

!cd {LLAMA_CPP_DIR} && \
    cmake -B build -DBUILD_SHARED_LIBS=OFF -DGGML_CUDA=OFF -DCMAKE_BUILD_TYPE=Release && \
    cmake --build build --config Release -j$(nproc) --target llama-quantize llama-cli

# Verify
for tool in ["llama-quantize", "llama-cli"]:
    path = f"{LLAMA_CPP_DIR}/build/bin/{tool}"
    print(f"{'✅' if os.path.exists(path) else '❌'} {tool}")

## Step 3: Download Sarvam-30B

In [ ]:
import os
from huggingface_hub import snapshot_download

if os.path.exists(f"{MODEL_DIR}/config.json"):
    print("✅ Model already downloaded, skipping.")
else:
    print("⬇️  Downloading sarvamai/sarvam-30b to Google Drive...")
    print("   This takes ~15-30 min depending on connection speed.")
    print("   If interrupted, re-run this cell — it resumes automatically.\n")
    try:
        snapshot_download(
            repo_id="sarvamai/sarvam-30b",
            local_dir=MODEL_DIR,
            ignore_patterns=["*.bin", "*.h5", "*.msgpack", "original/**"],
            resume_download=True,
        )
        print("\n✅ Download complete!")
    except Exception as e:
        print(f"\n❌ Download failed: {e}")
        print("\nTips:")
        print("  - Check Google Drive storage (need ~60GB free)")
        print("  - Re-run this cell to resume")
        print("  - If persistent, try: Runtime → Restart runtime, then re-run")
        raise

# Clean up local HF cache
!rm -rf /content/hf_cache 2>/dev/null

print(f"\nModel size: ", end="")
!du -sh {MODEL_DIR} 2>/dev/null
!ls {MODEL_DIR}/*.safetensors 2>/dev/null | wc -l | xargs -I {} echo "{} safetensor files"

## Step 4: Inspect Architecture

In [ ]:
import json

with open(f"{MODEL_DIR}/config.json") as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

print(f"\n{'='*50}")
print(f"Model type:     {config.get('model_type', 'unknown')}")
print(f"Architectures:  {config.get('architectures', [])}")

moe_keys = [k for k in config.keys() if any(x in k.lower() for x in ['expert', 'moe', 'router', 'routing'])]
print(f"\nMoE config:")
for k in moe_keys:
    print(f"  {k}: {config[k]}")
print(f"{'='*50}")

## Step 5: Check llama.cpp Support

In [ ]:
LLAMA_CPP_DIR = "/content/llama.cpp"

print("=== Sarvam/sigmoid in conversion script ===")
!grep -n -i 'sarvam\|sigmoid\|SarvamMoe' {LLAMA_CPP_DIR}/convert_hf_to_gguf.py || echo '❌ Not found'

print("\n=== Sarvam in gguf-py ===")
!grep -rn -i 'sarvam\|sigmoid' {LLAMA_CPP_DIR}/gguf-py/ 2>/dev/null | head -10 || echo '❌ Not found'

print("\n=== Sarvam in C++ source ===")
!grep -rn -i 'sarvam' {LLAMA_CPP_DIR}/src/ {LLAMA_CPP_DIR}/include/ 2>/dev/null | head -10 || echo '❌ Not found'

print("\n=== All supported architectures ===")
!grep -oP 'class \w+Model' {LLAMA_CPP_DIR}/convert_hf_to_gguf.py 2>/dev/null | sort -u || \
 grep 'class.*Model' {LLAMA_CPP_DIR}/convert_hf_to_gguf.py | head -40

## Step 6: Attempt GGUF Conversion

**This is the critical step.** If Sarvam's `model_type` isn't recognized, it fails here.

In [ ]:
import os, time

LLAMA_CPP_DIR = "/content/llama.cpp"
GGUF_F16 = f"{OUTPUT_DIR}/sarvam-30b-f16.gguf"

print("🔄 Attempting GGUF conversion...")
print(f"   Input:  {MODEL_DIR}")
print(f"   Output: {GGUF_F16}\n")

start = time.time()

!python3 {LLAMA_CPP_DIR}/convert_hf_to_gguf.py \
    {MODEL_DIR} \
    --outfile {GGUF_F16} \
    --outtype f16 \
    2>&1 | tee {OUTPUT_DIR}/conversion_log.txt

elapsed = time.time() - start
print(f"\n⏱️  Took {elapsed/60:.1f} minutes")

if os.path.exists(GGUF_F16):
    size_gb = os.path.getsize(GGUF_F16) / (1024**3)
    print(f"\n{'='*50}")
    print(f"✅ SUCCESS! GGUF created: {size_gb:.2f} GB")
    print(f"{'='*50}")
    CONVERSION_OK = True
else:
    print(f"\n{'='*50}")
    print(f"❌ CONVERSION FAILED")
    print(f"{'='*50}")
    print("See Step 8 for diagnosis.")
    CONVERSION_OK = False

## Step 7: Quantize (if conversion succeeded)

In [ ]:
import os, shutil

LLAMA_CPP_DIR = "/content/llama.cpp"
GGUF_F16 = f"{OUTPUT_DIR}/sarvam-30b-f16.gguf"
QUANTIZE = f"{LLAMA_CPP_DIR}/build/bin/llama-quantize"

if not os.path.exists(GGUF_F16):
    print("⏭️  No GGUF to quantize. Conversion didn't succeed.")
elif not os.path.exists(QUANTIZE):
    print("❌ llama-quantize not found. Re-run Step 2.")
else:
    _, _, free = shutil.disk_usage(OUTPUT_DIR)
    free_gb = free / (1024**3)
    print(f"Drive free: {free_gb:.1f} GB\n")

    methods = ["q8_0", "q4_k_m"] if free_gb > 40 else ["q4_k_m"]
    if free_gb <= 40:
        print("⚠️  Low disk — only doing Q4_K_M\n")

    for method in methods:
        out = f"{OUTPUT_DIR}/sarvam-30b-{method}.gguf"
        print(f"🔄 Quantizing to {method}...")
        !{QUANTIZE} {GGUF_F16} {out} {method}
        if os.path.exists(out):
            print(f"✅ {method}: {os.path.getsize(out)/(1024**3):.2f} GB\n")
        else:
            print(f"❌ {method} failed\n")

    # Clean up F16 to save space
    if os.path.exists(f"{OUTPUT_DIR}/sarvam-30b-q4_k_m.gguf"):
        print("🗑️  Removing F16 GGUF to save disk...")
        os.remove(GGUF_F16)

    print("\n📁 Output files:")
    for f in sorted(os.listdir(OUTPUT_DIR)):
        path = os.path.join(OUTPUT_DIR, f)
        if os.path.isfile(path):
            size = os.path.getsize(path)
            unit = f"{size/(1024**3):.2f} GB" if size > 1024*1024 else f"{size} bytes"
            print(f"  {f}: {unit}")

## Step 8: Upload to HuggingFace (optional)

In [ ]:
# Uncomment, set YOUR_USERNAME, and run to upload GGUFs to HuggingFace.

# from huggingface_hub import HfApi, login
# login()
# api = HfApi()
# api.create_repo("YOUR_USERNAME/sarvam-30b-GGUF", repo_type="model", exist_ok=True)
#
# import glob
# for gguf in glob.glob(f"{OUTPUT_DIR}/*.gguf"):
#     name = os.path.basename(gguf)
#     print(f"Uploading {name}...")
#     api.upload_file(
#         path_or_fileobj=gguf,
#         path_in_repo=name,
#         repo_id="YOUR_USERNAME/sarvam-30b-GGUF",
#         repo_type="model",
#     )
#     print(f"  ✅ Done")

print("ℹ️  Uncomment this cell and set YOUR_USERNAME to upload.")

## Step 9: Diagnosis (if conversion failed)

If Step 6 failed, this tells you exactly why.

In [ ]:
import json

LLAMA_CPP_DIR = "/content/llama.cpp"

with open(f"{MODEL_DIR}/config.json") as f:
    config = json.load(f)

model_type = config.get("model_type", "unknown")
archs = config.get("architectures", [])

# Check if llama.cpp knows this model type
import subprocess
result = subprocess.run(
    ["grep", "-c", "-i", model_type, f"{LLAMA_CPP_DIR}/convert_hf_to_gguf.py"],
    capture_output=True, text=True
)
count = int(result.stdout.strip()) if result.returncode == 0 else 0

print(f"{'='*60}")
print(f"DIAGNOSIS")
print(f"{'='*60}")
print(f"")
print(f"Sarvam model_type:    '{model_type}'")
print(f"Sarvam architectures: {archs}")
print(f"")

if count > 0:
    print(f"✅ '{model_type}' found {count} times in conversion script")
    print(f"   Conversion should work — check Step 6 logs for other errors.")
else:
    print(f"❌ '{model_type}' NOT found in convert_hf_to_gguf.py")
    print(f"   This is the blocker. llama.cpp doesn't know this architecture.")
    print(f"")
    print(f"   What needs to happen:")
    print(f"   1. llama.cpp PR #20275 needs to merge (adds sarvam_moe arch)")
    print(f"   2. Or: patch convert_hf_to_gguf.py using vLLM PR #33942 as reference")
    print(f"   3. Key blocker: sigmoid routing in MoE layer")

print(f"")
print(f"{'='*60}")
print(f"References:")
print(f"  vLLM PR #33942 (merged):    github.com/vllm-project/vllm/pull/33942")
print(f"  llama.cpp PR #20275 (open): github.com/ggml-org/llama.cpp/pull/20275")
print(f"  Analysis: mtrajan.substack.com/p/sarvam-open-is-not-sovereign")
print(f"{'='*60}")